In [ ]:
!pip install datasets evaluate scikit-learn transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np

ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

In [ ]:
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=256)

tokenized_datasets = ds.map(tokenize_function, batched=True)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "precision": precision.compute(predictions=predictions, references=labels, average="weighted")["precision"],
        "recall": recall.compute(predictions=predictions, references=labels, average="weighted")["recall"],
        "f1": f1.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    }

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

training_args = TrainingArguments(
    output_dir="./results",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

trainer.train()

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Step,Training Loss
50,0.300500
100,0.236100
150,0.203300
200,0.244200
250,0.225700
300,0.255800
350,0.173800
400,0.171900
450,0.148700
500,0.143100


TrainOutput(global_step=5335, training_loss=0.11176720089868283, metrics={'train_runtime': 1027.0567, 'train_samples_per_second': 41.526, 'train_steps_per_second': 5.194, 'total_flos': 2824968031334400.0, 'train_loss': 0.11176720089868283, 'epoch': 5.0})

In [ ]:
metrics = trainer.evaluate()
print("\nFinal Evaluation:", metrics)


Final Evaluation: {'eval_loss': 1.091781735420227, 'eval_accuracy': 0.8480300187617261, 'eval_precision': 0.8481084411583487, 'eval_recall': 0.8480300187617261, 'eval_f1': 0.8480214592727926, 'eval_runtime': 7.6446, 'eval_samples_per_second': 139.444, 'eval_steps_per_second': 17.529, 'epoch': 5.0}


In [ ]:
model.save_pretrained("./trained_model")
tokenizer.save_pretrained("./trained_model")

('./trained_model/tokenizer_config.json',
 './trained_model/special_tokens_map.json',
 './trained_model/vocab.txt',
 './trained_model/added_tokens.json',
 './trained_model/tokenizer.json')

In [ ]:
import shutil

shutil.make_archive("trained_model", 'zip', "trained_model")

'/content/trained_model.zip'

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

id2label = {
    0: "Negative",
    1: "Positive",
}
texts = [
"I hate this movie so much, it reminds me of my highschool bully",
          "I love this movie so much, it reminds me of my friend from highschool"
]

inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(device)

outputs = model(**inputs)
predictions = outputs.logits.argmax(-1)

for text, pred in zip(texts, predictions):
    label = id2label[pred.item()]
    print(f"Review: {text}\nPrediction: {label}\n")


Review: I hate this movie so much, it reminds me of my highschool bully
Prediction: Negative

Review: I love this movie so much, it reminds me of my friend from highschool
Prediction: Positive



In [ ]:
from torch.utils.data import DataLoader

model.config.output_hidden_states = True

def get_embeddings(texts, tokenizer, model, device, batch_size=16):
    model.eval()
    all_embeddings = []
    dataloader = DataLoader(texts, batch_size=batch_size)
    with torch.no_grad():
        for batch in dataloader:
            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
            outputs = model(**inputs)
            embeddings = outputs.hidden_states[-1][:,0,:].cpu().numpy()
            all_embeddings.append(embeddings)
    return np.vstack(all_embeddings)

train_texts = list(tokenized_datasets["train"]["text"])
train_embeddings = get_embeddings(train_texts, tokenizer, model, device)



In [ ]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=5, metric="euclidean")
nn.fit(train_embeddings)

def search_similar(query, tokenizer, model, device, nn, texts, k=5):
    query_emb = get_embeddings([query], tokenizer, model, device)
    D, I = nn.kneighbors(query_emb, n_neighbors=k)
    return [(texts[i], D[0][j]) for j, i in enumerate(I[0])]


query = "I feel like I wasted my time by watching this movie"
results = search_similar(query, tokenizer, model, device, nn, train_texts, k=5)
for text, dist in results:
    print(f"Distance: {dist:.4f} | Text: {text}")



Distance: 4.3100 | Text: the whole movie is simply a lazy exercise in bad filmmaking that asks you to not only suspend your disbelief but your intelligence as well .
Distance: 4.4696 | Text: it's a bad action movie because there's no rooting interest and the spectacle is grotesque and boring .
Distance: 4.5893 | Text: it's mindless junk like this that makes you appreciate original romantic comedies like punch-drunk love .
Distance: 4.5943 | Text: it's lost the politics and the social observation and become just another situation romance about a couple of saps stuck in an inarticulate screenplay .
Distance: 4.5999 | Text: a loud , witless mess that has none of the charm and little of the intrigue from the tv series .
